# Notebook 7: Common Kernels

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 3 hours | **Week 10 — Support Vector Machines & Kernel Methods**

---

## Why This Matters

The kernel trick is what transforms SVMs from simple linear classifiers into one of the most powerful ML algorithms ever devised. Without kernels, SVMs can only draw straight lines. With the right kernel, they can carve out arbitrarily complex decision boundaries — while the math stays clean and tractable.

In real-world email classification, your data is never perfectly linearly separable. Spam email might share vocabulary with promotions. Ham might look like newsletters. You need a classifier that can learn subtle, curved boundaries in high-dimensional feature spaces. Kernels are the tool that makes this possible.

---

## Real-World Analogy: Camera Lenses

Think of kernels as **different lenses for a camera**:

- **Linear lens (standard kit lens):** Captures flat, straight-edged scenes faithfully. Great for landscapes, architecture. Fails when you need to separate overlapping regions.
- **Polynomial lens (zoom lens):** Reveals curves, arcs, and combinations. Lets you capture the interplay between features at multiple scales.
- **RBF lens (macro/microscope lens):** Extremely sensitive to *local* similarity. Gets up close and personal — a point only influences its immediate neighbourhood. Perfect when clusters are compact and well-separated.
- **Sigmoid lens (experimental artistic lens):** Inspired by neural network activations. Rarely the right choice, but useful to know about.

**Choosing the right kernel = choosing the right lens for your data.**  
The wrong lens gives you a blurry, unusable picture. The right one brings everything into sharp focus.

---

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain the geometric intuition behind linear, polynomial, RBF, and sigmoid kernels
2. Implement all four kernels from scratch using NumPy
3. Visualise how kernel choice and hyperparameters affect decision boundaries
4. Select the appropriate kernel for a given dataset (including email spam classification)
5. Tune `gamma` and `degree` using grid search and interpret the resulting accuracy heatmap

## Section 0 — Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.svm import SVC
from sklearn.datasets import make_circles, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import DecisionBoundaryDisplay

np.random.seed(42)
sns.set_theme(style='whitegrid')

# Consistent colour palette — maps to email classes used throughout the module
COLORS = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A']
KERNEL_COLORS = {'linear': '#E63946', 'poly': '#457B9D', 'rbf': '#2A9D8F', 'sigmoid': '#E9C46A'}

print('Libraries loaded successfully.')
print(f'NumPy {np.__version__} | sklearn loaded')

## Section 1 — The Four Kernels: Concept & Formula

A **kernel function** K(x, z) computes the dot product of two points *after* an implicit feature mapping φ:

$$K(\mathbf{x}, \mathbf{z}) = \langle \phi(\mathbf{x}), \phi(\mathbf{z}) \rangle$$

The genius: you never need to compute φ explicitly. The kernel does the inner product directly in the transformed space.

---

### 1.1 Linear Kernel
$$K(\mathbf{x}, \mathbf{z}) = \mathbf{x} \cdot \mathbf{z}$$

- **Feature mapping:** φ(x) = x (identity — no transformation)
- **Decision boundary:** hyperplane in original space
- **When to use:** High-dimensional, sparse data (TF-IDF text, genomics). When you have more features than samples. When you expect a linear relationship.
- **Hyperparameters:** Only `C` (regularisation). Very fast to train.

### 1.2 Polynomial Kernel
$$K(\mathbf{x}, \mathbf{z}) = (\gamma\, \mathbf{x} \cdot \mathbf{z} + r)^d$$

- **Parameters:** degree `d`, scale `gamma` (γ), bias `coef0` (r)
- **d = 1:** Reduces to linear kernel (with bias term)
- **d = 2:** Captures all pairwise feature interactions
- **Implicit space size:** O(p^d) features for p input dimensions
- **When to use:** Image classification, NLP tasks where word interactions matter
- **Risk:** High degree → exponentially large feature space → overfitting with small datasets

### 1.3 RBF (Radial Basis Function) / Gaussian Kernel
$$K(\mathbf{x}, \mathbf{z}) = \exp\left(-\gamma \|\mathbf{x} - \mathbf{z}\|^2\right)$$

- **Equivalent Gaussian width:** σ = 1/√(2γ)
- **Implicit space:** *Infinite*-dimensional Hilbert space
- **γ large:** Each support vector influences only nearby points → very complex, wiggly boundary → risk of overfitting
- **γ small:** Each support vector influences a wide region → smooth boundary → risk of underfitting
- **sklearn default:** `gamma='scale'` → γ = 1 / (n_features × X.var())
- **When to use:** General-purpose. First choice when the decision boundary is clearly non-linear and you don't know what to use.

### 1.4 Sigmoid Kernel
$$K(\mathbf{x}, \mathbf{z}) = \tanh(\gamma\, \mathbf{x} \cdot \mathbf{z} + r)$$

- **Inspired by:** Neural network activation function (tanh)
- **Validity:** NOT always a positive semi-definite (PSD) kernel. Only valid for certain γ and r values.
- **Practice:** Rarely outperforms linear or RBF. Included here for completeness and historical context.

---

| Kernel | Formula | Parameters | Best For |
|--------|---------|------------|----------|
| Linear | x·z | C | High-dim sparse (text) |
| Polynomial | (γx·z + r)^d | C, degree, gamma, coef0 | Image, NLP interactions |
| RBF | exp(-γ‖x-z‖²) | C, gamma | General non-linear |
| Sigmoid | tanh(γx·z + r) | C, gamma, coef0 | Rarely used |

## Section 2 — Implementing Kernels from Scratch

Before using sklearn's black-box implementation, let's build every kernel ourselves using NumPy.  
This reveals the exact computation happening inside the SVM solver.

In [ ]:
def linear_kernel(X, Z):
    """
    Linear kernel: K(x, z) = x . z
    Simple dot product — no feature transformation.
    """
    return X @ Z.T


def polynomial_kernel(X, Z, degree=2, gamma=1.0, coef0=1.0):
    """
    Polynomial kernel: K(x, z) = (gamma * x.z + coef0)^degree
    Implicitly maps to degree-d polynomial feature space.
    """
    return (gamma * X @ Z.T + coef0) ** degree


def rbf_kernel(X, Z, gamma=1.0):
    """
    RBF/Gaussian kernel: K(x, z) = exp(-gamma * ||x - z||^2)

    Efficient computation using the identity:
        ||x - z||^2 = ||x||^2 + ||z||^2 - 2 * x.z
    Avoids building a full pairwise distance matrix loop.
    """
    X_sq = np.sum(X ** 2, axis=1, keepdims=True)   # (n, 1)
    Z_sq = np.sum(Z ** 2, axis=1, keepdims=True)   # (m, 1)
    sq_dists = X_sq + Z_sq.T - 2 * (X @ Z.T)       # (n, m)
    return np.exp(-gamma * sq_dists)


def sigmoid_kernel(X, Z, gamma=0.01, coef0=0.0):
    """
    Sigmoid kernel: K(x, z) = tanh(gamma * x.z + coef0)
    Not always PSD — use with caution.
    """
    return np.tanh(gamma * X @ Z.T + coef0)


# ── Quick sanity check on toy data ──────────────────────────────────────────
np.random.seed(42)
X_toy = np.random.randn(5, 3)
Z_toy = np.random.randn(4, 3)

print('Linear kernel shape:     ', linear_kernel(X_toy, Z_toy).shape)       # (5, 4)
print('Polynomial kernel shape: ', polynomial_kernel(X_toy, Z_toy).shape)   # (5, 4)
print('RBF kernel shape:        ', rbf_kernel(X_toy, Z_toy).shape)          # (5, 4)
print('Sigmoid kernel shape:    ', sigmoid_kernel(X_toy, Z_toy).shape)      # (5, 4)

# RBF diagonal (self-similarity) should be 1 when X == Z
K_self = rbf_kernel(X_toy, X_toy, gamma=1.0)
print('\nRBF self-similarity (diagonal should be 1.0):', np.diag(K_self).round(6))

## Section 3 — Creating Three Test Datasets

To stress-test each kernel, we need datasets with different geometric structures:

- **Dataset A (Linear):** Two Gaussian blobs — linearly separable. Represents clean spam vs ham.
- **Dataset B (Circular):** Concentric rings — requires curved boundary. Represents promotions embedded inside ham.
- **Dataset C (Spiral):** Interleaved spirals — extremely non-linear. Represents newsletter vs spam with overlapping vocabulary patterns.

In [ ]:
np.random.seed(42)

# ── Dataset A: Linearly separable ─────────────────────────────────────────
n = 200
X_lin = np.vstack([
    np.random.randn(n // 2, 2) + np.array([-2, -2]),
    np.random.randn(n // 2, 2) + np.array([2, 2])
])
y_lin = np.array([0] * (n // 2) + [1] * (n // 2))

# ── Dataset B: Circular (concentric rings) ────────────────────────────────
X_circ, y_circ = make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=42)

# ── Dataset C: Spiral ─────────────────────────────────────────────────────
def make_spiral(n_samples=300, noise=0.3, random_state=42):
    rng = np.random.RandomState(random_state)
    n = n_samples // 2
    theta = np.linspace(0, 4 * np.pi, n)
    r = np.linspace(0.5, 4, n)
    X1 = np.c_[r * np.cos(theta), r * np.sin(theta)] + rng.randn(n, 2) * noise
    X2 = np.c_[r * np.cos(theta + np.pi), r * np.sin(theta + np.pi)] + rng.randn(n, 2) * noise
    X = np.vstack([X1, X2])
    y = np.array([0] * n + [1] * n)
    return X, y

X_spiral, y_spiral = make_spiral(n_samples=300, noise=0.25, random_state=42)

# Scale all datasets (SVM is sensitive to feature scale)
scaler = StandardScaler()
X_lin    = scaler.fit_transform(X_lin)
X_circ   = scaler.fit_transform(X_circ)
X_spiral = scaler.fit_transform(X_spiral)

datasets = [
    ('Linear (Spam vs Ham)',   X_lin,    y_lin),
    ('Circular (Promo vs Ham)', X_circ,  y_circ),
    ('Spiral (Newsletter vs Spam)', X_spiral, y_spiral),
]

# Quick overview
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (title, X, y) in zip(axes, datasets):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='k', linewidths=0.4, alpha=0.8, s=40)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
plt.suptitle('Three Test Datasets — Increasing Geometric Complexity', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Dataset sizes:', [X.shape for _, X, _ in datasets])

## Section 4 — All Kernels × All Datasets: 3×4 Decision Boundary Grid

Now the main event: apply every kernel to every dataset.  
Read this grid as a **camera lens vs subject** matrix — the right combination gives a sharp, accurate boundary.

In [ ]:
kernels = [
    ('linear',  'Linear',     {'kernel': 'linear',  'C': 1.0}),
    ('poly',    'Polynomial\n(degree=3)', {'kernel': 'poly', 'C': 1.0, 'degree': 3, 'gamma': 'scale', 'coef0': 1.0}),
    ('rbf',     'RBF',        {'kernel': 'rbf',     'C': 1.0, 'gamma': 'scale'}),
    ('sigmoid', 'Sigmoid',    {'kernel': 'sigmoid', 'C': 1.0, 'gamma': 0.5, 'coef0': 0.0}),
]

fig, axes = plt.subplots(3, 4, figsize=(18, 13))
fig.suptitle('Kernel × Dataset Decision Boundary Grid\n(Each cell = one SVM fit)', 
             fontsize=14, fontweight='bold', y=1.01)

xx_range = np.linspace(-3.5, 3.5, 300)
yy_range = np.linspace(-3.5, 3.5, 300)
XX, YY = np.meshgrid(xx_range, yy_range)
grid_points = np.c_[XX.ravel(), YY.ravel()]

results_table = []

for row_idx, (ds_title, X, y) in enumerate(datasets):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    for col_idx, (k_key, k_label, k_params) in enumerate(kernels):
        ax = axes[row_idx, col_idx]

        clf = SVC(**k_params)
        clf.fit(X_tr, y_tr)
        train_acc = accuracy_score(y_tr, clf.predict(X_tr))
        test_acc  = accuracy_score(y_te, clf.predict(X_te))

        Z = clf.predict(grid_points).reshape(XX.shape)
        ax.contourf(XX, YY, Z, alpha=0.25, cmap='RdYlBu')
        ax.contour(XX, YY, Z, colors='k', linewidths=0.8)
        ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu',
                   edgecolors='k', linewidths=0.3, alpha=0.9, s=25)

        ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
        ax.set_xticks([]); ax.set_yticks([])

        title_color = '#2A9D8F' if test_acc >= 0.85 else ('#E9C46A' if test_acc >= 0.70 else '#E63946')
        ax.set_title(f'{k_label}\nTrain: {train_acc:.2f} | Val: {test_acc:.2f}',
                     fontsize=9, color=title_color, fontweight='bold')

        if col_idx == 0:
            ax.set_ylabel(ds_title, fontsize=9, fontweight='bold')

        results_table.append({'Dataset': ds_title, 'Kernel': k_label.replace('\n', ' '),
                               'Train Acc': round(train_acc, 3), 'Val Acc': round(test_acc, 3)})

plt.tight_layout()
plt.show()

# Summary table
df_results = pd.DataFrame(results_table)
print('\nSummary (green = good, yellow = OK, red = poor):')
print(df_results.to_string(index=False))

### Reading the Grid

- **Row 1 (Linear dataset):** All kernels should work. Linear wins on simplicity; poly/RBF may slightly overfit.
- **Row 2 (Circular):** Linear kernel fails (it can only draw a line). RBF should shine here — it naturally handles circular structure.
- **Row 3 (Spiral):** The hardest. Even RBF struggles without careful gamma tuning. Sigmoid often breaks down entirely.

**Key insight:** A kernel that fits all datasets perfectly doesn't exist (no free lunch). Choose based on *your* data's structure.

## Section 5 — Gram Matrix Visualisation

The **Gram matrix** (kernel matrix) K is an n×n matrix where K[i,j] = K(x_i, x_j).  
It encodes all pairwise similarities under the kernel's implied geometry.

- **Block structure** (bright blocks on diagonal) = good class separation in kernel space
- **Uniform noise** = kernel can't distinguish classes
- This is what the SVM solver actually works with internally

In [ ]:
# Use the circular dataset, first 80 points (sorted by class for clarity)
np.random.seed(42)
n_gram = 80
idx_0 = np.where(y_circ == 0)[0][:n_gram // 2]
idx_1 = np.where(y_circ == 1)[0][:n_gram // 2]
idx_sorted = np.concatenate([idx_0, idx_1])
X_gram = X_circ[idx_sorted]
y_gram = y_circ[idx_sorted]

kernel_funcs = [
    ('Linear\nK(x,z) = x·z',              linear_kernel(X_gram, X_gram)),
    ('Polynomial (d=2)\nK=(γx·z+1)²',    polynomial_kernel(X_gram, X_gram, degree=2)),
    ('RBF (γ=1)\nK=exp(-γ‖x-z‖²)',       rbf_kernel(X_gram, X_gram, gamma=1.0)),
    ('Sigmoid\nK=tanh(γx·z+r)',           sigmoid_kernel(X_gram, X_gram, gamma=0.5, coef0=0.0)),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle('Gram (Kernel) Matrix for 80 Circular-Dataset Points\n'
             '(First 40 = class 0 / inner ring | Last 40 = class 1 / outer ring)',
             fontsize=12, fontweight='bold')

for ax, (title, K) in zip(axes, kernel_funcs):
    # Normalise for fair visual comparison
    K_norm = (K - K.min()) / (K.max() - K.min() + 1e-10)
    sns.heatmap(K_norm, ax=ax, cmap='viridis', cbar=True,
                xticklabels=False, yticklabels=False, square=True)
    ax.set_title(title, fontsize=9, fontweight='bold')
    # Draw class boundary lines
    ax.axhline(n_gram // 2, color='red', linewidth=1.5, linestyle='--')
    ax.axvline(n_gram // 2, color='red', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Sample index'); ax.set_ylabel('Sample index')

plt.tight_layout()
plt.show()

print('Interpretation guide:')
print('  Bright blocks on diagonal (top-left, bottom-right) = high within-class similarity')
print('  Dark off-diagonal blocks = low between-class similarity = good separation')
print('  Linear kernel: class structure barely visible (linear kernel cannot separate circles)')
print('  RBF kernel:    strong block structure = RBF correctly captures circular geometry')

## Section 6 — RBF Gamma Sweep

γ is the single most important hyperparameter for the RBF kernel.  
Think of γ as controlling the **radius of influence** of each support vector:

- **γ large:** Each SV is a sharp spike — it only influences a tiny neighbourhood → complex, wiggly boundaries → **overfitting**
- **γ small:** Each SV spreads its influence widely → smooth, gentle boundaries → **underfitting**

The sweet spot is where the boundary is complex enough to capture the true structure but not so complex that it memorises noise.

In [ ]:
gammas = [0.01, 0.1, 1.0, 10.0, 100.0]

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_circ, y_circ, test_size=0.25, random_state=42, stratify=y_circ
)

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('RBF Kernel — Gamma Sweep on Circular Dataset\n'
             'γ controls radius of influence: small γ = smooth, large γ = spiky',
             fontsize=12, fontweight='bold')

xx_c = np.linspace(X_circ[:, 0].min() - 0.5, X_circ[:, 0].max() + 0.5, 300)
yy_c = np.linspace(X_circ[:, 1].min() - 0.5, X_circ[:, 1].max() + 0.5, 300)
XX_c, YY_c = np.meshgrid(xx_c, yy_c)
grid_c = np.c_[XX_c.ravel(), YY_c.ravel()]

train_accs, val_accs, n_svs = [], [], []

for ax, gamma in zip(axes, gammas):
    clf = SVC(kernel='rbf', C=1.0, gamma=gamma)
    clf.fit(X_tr_c, y_tr_c)
    tr_acc = accuracy_score(y_tr_c, clf.predict(X_tr_c))
    te_acc = accuracy_score(y_te_c, clf.predict(X_te_c))
    train_accs.append(tr_acc); val_accs.append(te_acc); n_svs.append(clf.n_support_.sum())

    Z = clf.predict(grid_c).reshape(XX_c.shape)
    ax.contourf(XX_c, YY_c, Z, alpha=0.3, cmap='RdYlBu')
    ax.contour(XX_c, YY_c, Z, colors='k', linewidths=0.8)
    ax.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ,
               cmap='RdYlBu', edgecolors='k', linewidths=0.3, s=25, alpha=0.9)
    # Mark support vectors
    sv = clf.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=90, facecolors='none',
               edgecolors='gold', linewidths=1.5, label='SVs')

    color = '#2A9D8F' if te_acc >= 0.88 else ('#E9C46A' if te_acc >= 0.75 else '#E63946')
    ax.set_title(f'γ = {gamma}\nTrain: {tr_acc:.2f} | Val: {te_acc:.2f}\nSVs: {n_svs[-1]}',
                 fontsize=9, color=color, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

# Accuracy vs gamma curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
log_g = np.log10(gammas)
ax1.plot(log_g, train_accs, 'o-', color='#E63946', label='Train Accuracy', linewidth=2)
ax1.plot(log_g, val_accs,   's-', color='#457B9D', label='Val Accuracy',   linewidth=2)
ax1.set_xlabel('log₁₀(γ)', fontsize=11); ax1.set_ylabel('Accuracy', fontsize=11)
ax1.set_title('Bias–Variance Tradeoff vs Gamma', fontsize=11, fontweight='bold')
ax1.legend(); ax1.set_xticks(log_g); ax1.set_xticklabels([str(g) for g in gammas])
ax1.set_ylim(0.4, 1.05)
ax1.axvspan(log_g[0], log_g[1], alpha=0.1, color='blue', label='Underfitting zone')
ax1.axvspan(log_g[3], log_g[4], alpha=0.1, color='red',  label='Overfitting zone')

ax2.bar(log_g, n_svs, color='#2A9D8F', edgecolor='k', width=0.4)
ax2.set_xlabel('log₁₀(γ)', fontsize=11); ax2.set_ylabel('Number of Support Vectors', fontsize=11)
ax2.set_title('Support Vector Count vs Gamma\n(More SVs = More Complex Model)', fontsize=11, fontweight='bold')
ax2.set_xticks(log_g); ax2.set_xticklabels([str(g) for g in gammas])

plt.tight_layout()
plt.show()

## Section 7 — Polynomial Degree Sweep

For the polynomial kernel, `degree` controls the complexity of the feature space:
- **d=1:** Linear boundary (same as linear kernel with bias)
- **d=2:** Can capture quadratic curves, ellipses, parabolas
- **d=3+:** Increasingly complex boundaries; risk of overfitting grows rapidly with small datasets

In [ ]:
degrees = [1, 2, 3, 5, 10]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Polynomial Kernel — Degree Sweep on Circular Dataset\n'
             '(coef0=1, gamma=1, C=1)',
             fontsize=12, fontweight='bold')

poly_train_accs, poly_val_accs = [], []

for ax, d in zip(axes, degrees):
    clf = SVC(kernel='poly', C=1.0, degree=d, gamma=1.0, coef0=1.0)
    clf.fit(X_tr_c, y_tr_c)
    tr_acc = accuracy_score(y_tr_c, clf.predict(X_tr_c))
    te_acc = accuracy_score(y_te_c, clf.predict(X_te_c))
    poly_train_accs.append(tr_acc); poly_val_accs.append(te_acc)

    # Implicit feature space size for 2D input
    # C(p+d, d) where p=2
    feature_count = int(np.math.comb(2 + d, d)) if d <= 10 else '...'

    Z = clf.predict(grid_c).reshape(XX_c.shape)
    ax.contourf(XX_c, YY_c, Z, alpha=0.3, cmap='RdYlBu')
    ax.contour(XX_c, YY_c, Z, colors='k', linewidths=0.8)
    ax.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ,
               cmap='RdYlBu', edgecolors='k', linewidths=0.3, s=25, alpha=0.9)

    color = '#2A9D8F' if te_acc >= 0.88 else ('#E9C46A' if te_acc >= 0.75 else '#E63946')
    ax.set_title(f'degree = {d}\nTrain: {tr_acc:.2f} | Val: {te_acc:.2f}\nImpl. features: {feature_count}',
                 fontsize=9, color=color, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

# Feature space growth
print('\nImplicit feature space size (2D input → degree d polynomial):')
for d in degrees:
    from math import comb
    # For general p=10000 text feature space
    print(f'  d={d:2d}: {comb(2 + d, d):>8,} features (2D input) | '
          f'~{(10000**d / np.math.factorial(d)):,.0f} features (p=10k text)' if d <= 2 else
          f'  d={d:2d}: {comb(2 + d, d):>8,} features (2D input) | [too large to write for p=10k]')

## Section 8 — Grid Search: C × Gamma for RBF Kernel

In practice, you tune both `C` (regularisation strength) and `gamma` together.  
They interact: a large C combined with large gamma is a recipe for extreme overfitting.  
We visualise the full grid as a **heatmap of validation accuracy**.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Use spiral dataset — hardest problem, most sensitive to hyperparameters
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_spiral, y_spiral, test_size=0.25, random_state=42, stratify=y_spiral
)

param_grid = {
    'C':     [0.01, 0.1, 1, 10, 100, 1000],
    'gamma': [0.001, 0.01, 0.1, 1, 10, 100],
}

grid_search = GridSearchCV(
    SVC(kernel='rbf'),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_tr_s, y_tr_s)

# Reshape results into matrix form
C_vals     = param_grid['C']
gamma_vals = param_grid['gamma']
scores = grid_search.cv_results_['mean_test_score'].reshape(len(C_vals), len(gamma_vals))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
ax = axes[0]
sns.heatmap(
    scores,
    ax=ax,
    annot=True, fmt='.2f', cmap='YlGn',
    xticklabels=gamma_vals,
    yticklabels=C_vals,
    linewidths=0.5, linecolor='grey',
    vmin=0.5, vmax=1.0
)
ax.set_xlabel('gamma', fontsize=12, fontweight='bold')
ax.set_ylabel('C (regularisation)', fontsize=12, fontweight='bold')
ax.set_title('Grid Search: C × Gamma — 5-Fold CV Accuracy\nRBF Kernel on Spiral Dataset',
             fontsize=11, fontweight='bold')

# Best model decision boundary
best_C     = grid_search.best_params_['C']
best_gamma = grid_search.best_params_['gamma']
best_score = grid_search.best_score_

best_clf = SVC(kernel='rbf', C=best_C, gamma=best_gamma)
best_clf.fit(X_tr_s, y_tr_s)

ax2 = axes[1]
xx_s = np.linspace(X_spiral[:, 0].min() - 0.3, X_spiral[:, 0].max() + 0.3, 300)
yy_s = np.linspace(X_spiral[:, 1].min() - 0.3, X_spiral[:, 1].max() + 0.3, 300)
XX_s, YY_s = np.meshgrid(xx_s, yy_s)
Z_best = best_clf.predict(np.c_[XX_s.ravel(), YY_s.ravel()]).reshape(XX_s.shape)

ax2.contourf(XX_s, YY_s, Z_best, alpha=0.35, cmap='RdYlBu')
ax2.contour(XX_s,  YY_s, Z_best, colors='k', linewidths=0.8)
ax2.scatter(X_spiral[:, 0], X_spiral[:, 1], c=y_spiral,
            cmap='RdYlBu', edgecolors='k', linewidths=0.3, s=25, alpha=0.9)
test_acc = accuracy_score(y_te_s, best_clf.predict(X_te_s))
ax2.set_title(f'Best Model Decision Boundary\nC={best_C}, γ={best_gamma}\n'
              f'CV Score: {best_score:.3f} | Test Acc: {test_acc:.3f}',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('Feature 1'); ax2.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print(f'Best hyperparameters: C={best_C}, gamma={best_gamma}')
print(f'Best CV accuracy: {best_score:.4f}')
print(f'Test accuracy:    {test_acc:.4f}')

## Section 9 — Kernel Choice Guide: Email Classification

Now let's apply our kernel knowledge to the actual problem: **email spam classification with TF-IDF features**.

In [ ]:
# Simulate a high-dimensional sparse TF-IDF email dataset
# Realistic: 5000 emails, 20,000 TF-IDF vocabulary features, sparse
from scipy.sparse import random as sparse_random
from sklearn.linear_model import LinearSVC  # Fast linear SVM

np.random.seed(42)
n_emails    = 1000  # Keep small for demo; real-world: 50k-1M
n_features  = 5000  # TF-IDF vocabulary
sparsity    = 0.02  # Only 2% of words appear in each email

# Simulate sparse TF-IDF matrix
X_email = sparse_random(n_emails, n_features, density=sparsity, 
                        format='csr', random_state=42).toarray()
X_email = X_email.astype(np.float32)
y_email = np.random.randint(0, 2, n_emails)  # 0=ham, 1=spam

X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(
    X_email, y_email, test_size=0.2, random_state=42
)

import time

print('Comparing kernel performance on simulated TF-IDF email data')
print(f'Dataset: {n_emails} emails × {n_features} TF-IDF features (density={sparsity})')
print('=' * 65)

kernel_configs = [
    ('Linear SVC (LibLinear)',   LinearSVC(C=1.0, max_iter=2000)),
    ('SVC linear kernel',        SVC(kernel='linear', C=1.0)),
    ('SVC RBF kernel',           SVC(kernel='rbf', C=1.0, gamma='scale')),
    ('SVC poly d=2',             SVC(kernel='poly', C=1.0, degree=2, gamma='scale')),
]

results = []
for name, clf in kernel_configs:
    t0 = time.time()
    clf.fit(X_tr_e, y_tr_e)
    elapsed = time.time() - t0
    acc = accuracy_score(y_te_e, clf.predict(X_te_e))
    print(f'{name:<30}  Acc: {acc:.3f}  Train time: {elapsed:.3f}s')
    results.append({'Kernel': name, 'Accuracy': acc, 'Train Time (s)': round(elapsed, 3)})

print()
print('VERDICT for text/email classification (TF-IDF):')
print('  USE: Linear kernel (LinearSVC or SVC with kernel="linear")')
print('  REASONS:')
print('    1. TF-IDF already lives in very high-dimensional space (p >> n)')
print('       Linear boundaries are sufficient — curse of dimensionality works IN your favour')
print('    2. Linear kernel trains in O(n×p) — orders of magnitude faster than RBF')
print('    3. RBF gamma computation is O(n²) — unscalable for 50k+ documents')
print('    4. LinearSVC uses liblinear (not libsvm) — handles sparse matrices natively')
print('    5. Interpretable: feature weights w_i tell you which words drive the decision')

## Section 10 — Quick Reference: Kernel Decision Tree

```
Is your data high-dimensional and sparse? (TF-IDF, bag-of-words, genomics)
   YES → Linear kernel  (fast, interpretable, scales to millions of features)
   NO  ↓

Do you know the data has polynomial structure? (image pixels, explicit interactions)
   YES → Polynomial kernel  (start with degree=2, tune gamma and coef0)
   NO  ↓

Is the decision boundary non-linear? (circles, blobs, curves)
   YES → RBF kernel  (default choice for low-to-medium dimensional non-linear data)
         Start with gamma='scale', tune with grid search
   NO  → Linear kernel

Sigmoid kernel: avoid unless you have a specific reason (e.g. reproducing old results)
```

**Rule of thumb:** When in doubt, try linear first, then RBF.  
**Never** skip scaling (StandardScaler) — all kernels are distance-based.

## Self-Check Questions

Test your understanding before moving on. Try answering each question yourself, then reveal the answer.

---

**Question 1:**  
An RBF SVM with γ = 1000 on a 2D dataset gives **training accuracy = 100%**, **validation accuracy = 52%** (barely above random). What is happening, and how would you fix it?

<details>
<summary>Show Answer</summary>

**What is happening: Severe overfitting due to γ too large.**

With γ = 1000, the radius of influence of each support vector is extremely small (σ = 1/√(2×1000) ≈ 0.022 units). Each support vector becomes a tiny "spike" that only classifies points in its immediate neighbourhood. The model essentially memorises the training set by drawing tiny circles around every training point, achieving 100% training accuracy.

On unseen validation data, these spikes are useless — new points fall into regions that no support vector covers, and the classifier defaults to the wrong side.

**How to fix it:**
1. **Reduce γ dramatically** — try γ = 0.01, 0.1, 1 with cross-validation
2. **Use `gamma='scale'`** — sklearn computes a data-informed default: γ = 1/(n_features × X.var())
3. **Increase C simultaneously if needed** — but be careful, C and γ both control complexity
4. **Use GridSearchCV** on the C × γ grid to find the combination with best CV accuracy
5. **Always scale features** (StandardScaler) before tuning γ — γ is scale-sensitive

**Key insight:** The 48-point accuracy gap (100% vs 52%) is a textbook overfitting signal. The model has zero generalisation — it's just memorised coordinates.

</details>

---

**Question 2:**  
The polynomial kernel of degree d=2 in a p-dimensional input space creates O(p²) implicit features. For text classification with p = 10,000 words (TF-IDF vocabulary), approximately how many features does the degree-2 polynomial create? Is this practical?

<details>
<summary>Show Answer</summary>

**Feature count calculation:**

The degree-2 polynomial kernel K(x,z) = (γx·z + r)² implicitly maps to a space of all monomials of degree ≤ 2:
- **Degree-0:** 1 feature (constant)
- **Degree-1:** p features (x₁, x₂, ..., x_p)
- **Degree-2:** p(p+1)/2 features (xᵢxⱼ for all i ≤ j, including xᵢ²)

Total ≈ 1 + p + p(p+1)/2 = **C(p+2, 2)** features

For p = 10,000:
- C(10002, 2) = 10002 × 10001 / 2 = **50,015,001 ≈ 50 million features**

**Is this practical?**
- **Storage:** Explicit computation is infeasible — 50M features × millions of training points
- **The kernel trick saves you:** The polynomial kernel computes K(x,z) = (x·z + 1)² in O(p) time (just a dot product + arithmetic), **without** ever building the 50M-dimensional vectors explicitly
- **But the Gram matrix is n×n** — for n=100,000 emails, K is 10¹⁰ entries = 80 GB just to store
- **Verdict:** For p=10,000 text, the kernel trick makes polynomial computation feasible per training step, but the O(n²) Gram matrix bottleneck makes it unscalable for large text corpora
- **Use linear kernel instead** — text classification almost always works just as well or better with a linear kernel, and it trains 10–100× faster

</details>

---

**Question 3:**  
For email classification with TF-IDF features (p = 50,000 vocabulary words), should you use a linear or RBF kernel? List at least three practical reasons that make the linear kernel the better choice here.

<details>
<summary>Show Answer</summary>

**Use the Linear Kernel. Reasons:**

1. **High-dimensional space works in your favour:**  
   With p = 50,000 features, data is already in an extremely high-dimensional space. Bennet & Mangasarian's result tells us that high-dimensional spaces are more likely to be linearly separable. A linear boundary in this 50k-dim space is extremely expressive — spam and ham have very different word distributions.

2. **Training speed (O(n·p) vs O(n²·p)):**  
   LinearSVC (using liblinear) trains in approximately O(n·p) time. RBF SVM requires computing the full n×n Gram matrix, costing O(n²·p). For 100,000 emails and 50,000 features: linear ≈ 5×10⁹ ops vs RBF ≈ 5×10¹⁴ ops. The linear kernel is 100,000× faster.

3. **Memory efficiency:**  
   RBF kernel requires storing the n×n Gram matrix. For 100k emails: 10¹⁰ floats = 80 GB. Linear kernel only needs the weight vector w of size p = 50,000 — about 200 KB.

4. **Sparse data compatibility:**  
   TF-IDF matrices are extremely sparse (typically 99%+ zeros). LibLinear (linear SVM) exploits this sparsity directly, only computing non-zero terms. RBF kernel distance ‖x-z‖² destroys sparsity — you must densify the matrix, losing the computational advantage.

5. **Interpretability:**  
   A linear SVM learns weight vector w where wᵢ corresponds to word i. You can directly read off which words drive spam classification (e.g., w['free'] > 0, w['meeting'] < 0). RBF gives no interpretable feature weights.

6. **Empirical evidence:**  
   On benchmark text classification datasets (20 Newsgroups, Reuters, RCV1), linear SVMs match or exceed RBF SVMs — while training 100–1000× faster. This has been known since Joachims (1998), the original "Text Categorisation with SVMs" paper.

**Practical recommendation:** Use `sklearn.svm.LinearSVC` (not `SVC(kernel='linear')`) — it uses liblinear, which is specifically optimised for large-scale linear classification and handles sparse matrices natively.

</details>